# Aula 03 · Vídeo 4 — Lab E2E: Pipeline Zero Trust (Presidio + MCP Isolado + Foundry)

> **Caso condutor:** Banco **Aurora** / **Aurora Assist** · **Tipo:** conceitual **+** prático (E2E)
> **Doc 04:** seções 7, 9-12 e **Figura 1** (pipeline Zero Trust end-to-end)

## 🎯 Objetivos
- **Consumir** a stack (não treinar modelo): ver PII **tokenizada no ingress**, o LLM/ferramentas vendo **só tokens** e a **reidratação** no egress para o titular.
- Ver a ferramenta de saldo (**somente leitura**) responder e a tentativa de **transferência ser bloqueada**.
- Medir **detecção de PII** (recall/precision/F1, *leak rate*) e **latência**; gerar `reports/`.

> [!WARNING]
> ⚠️ **Uso ético.** Dados sintéticos; a tentativa de abuso de ferramenta serve apenas para validar o isolamento.

## 0. Suba a stack (uma vez) — ou rode offline

```bash
uv run task up                       # docker compose up -d (rag-api + mcp-server + presidio + ollama)
docker compose exec ollama ollama pull llama3.1:8b
uv run task seed-data                # datasets sinteticos (seed fixa)
```

Este notebook detecta se o **rag-api** está no ar (`AURORA_API_URL`). Se **não** estiver, executa o **mesmo pipeline in-process** (offline, determinístico), de modo que *Restart & Run All* sempre funcione.

In [ ]:
import os, sys, json
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

os.environ.setdefault('PRESIDIO_MODE', 'deterministic')
os.environ.setdefault('TOKEN_VAULT_BACKEND', 'memory')

import random
random.seed(42)

p = ROOT / 'data' / 'pii_samples' / 'pii_samples.jsonl'
if not p.exists():
    import subprocess
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'generate_datasets.py')], check=True)
print('ROOT =', ROOT, '| dataset:', p.exists())

In [ ]:
import httpx

API = os.getenv('AURORA_API_URL', 'http://localhost:8000')
try:
    LIVE = httpx.get(API + '/health', timeout=2.0).status_code == 200
except Exception:
    LIVE = False
print('Modo:', 'LIVE (rag-api no ar)' if LIVE else 'OFFLINE (pipeline in-process)')

# Componentes in-process (modo OFFLINE e metricas deterministicas).
from app.pii import DataProtectionPipeline
from src.security import (
    ReversibleAnonymizer, DeterministicAnalyzer, InMemoryTokenVault,
    ToolCatalog, ToolAuthorizer, AuditTrail, issue_capability, detect_pii, detection_confusion,
)

_anon = ReversibleAnonymizer(analyzer=DeterministicAnalyzer('pt'), vault=InMemoryTokenVault(secret='lab'), min_score=0.5)
_pipe = DataProtectionPipeline(anonymizer=_anon)
_catalog = ToolCatalog.from_yaml(ROOT / 'app' / 'policies' / 'tool_policy.yaml')
_authz = ToolAuthorizer(catalog=_catalog, secret='lab-cap', audit=AuditTrail())
_ACCOUNTS = {'[BR_ACCOUNT_TOKEN_demo0001]': 'R$ 12.340,55'}

In [ ]:
def tool_call(tool, scope, args, subject='cliente-anon', flow='authenticated'):
    """Chama a ferramenta MCP (LIVE) ou o autorizador in-process (OFFLINE)."""
    if LIVE:
        r = httpx.post(f'{API}/tools/{tool}', json={'args': args, 'flow': flow, 'subject': subject}, timeout=30.0)
        return {'status': r.status_code, 'body': r.json()}
    token = issue_capability(subject=subject, tool=tool, scopes={scope}, secret='lab-cap')
    res = _authz.authorize(token, tool, scope, actor=subject)
    if not res.allowed:
        return {'status': 403, 'body': {'allowed': False, 'reason': res.reason}}
    out = _ACCOUNTS.get(args.get('conta_token'), 'conta nao encontrada') if tool == 'consultar_saldo' else 'ok'
    return {'status': 200, 'body': {'allowed': True, 'output': out}}


def anonymize_call(message, flow='authenticated'):
    if LIVE:
        return httpx.post(f'{API}/anonymize', json={'message': message, 'flow': flow}, timeout=30.0).json()
    return _pipe.anonymize(message, flow=flow).to_dict()


def chat(message, flow='authenticated', subject='cliente-anon'):
    if LIVE:
        return httpx.post(f'{API}/chat', json={'message': message, 'flow': flow, 'subject': subject}, timeout=60.0).json()
    ingress = _pipe.anonymize(message, flow=flow)
    profile = _pipe.profile(flow)
    tool_trace = None
    if 'saldo' in message.lower() and 'saldo:read' in profile.required_tool_scopes:
        # OFFLINE: usamos um token de conta demo (LIVE deriva do ingress).
        tool_trace = tool_call('consultar_saldo', 'saldo:read', {'conta_token': '[BR_ACCOUNT_TOKEN_demo0001]'}, subject, flow)['body']
        saldo_token = _anon.vault.tokenize('SALDO', str(tool_trace.get('output', '')))
        answer_internal = f'O saldo solicitado e {saldo_token}.'
    else:
        answer_internal = 'Posso ajudar com produtos e servicos do Banco Aurora.'
    delivered = _pipe.deanonymize(answer_internal, flow=flow)
    return {'answer': delivered, 'answer_internal': answer_internal, 'flow': flow,
            'tool': tool_trace, 'ingress': ingress.to_dict(), 'rehydrated': profile.rehydrate_on_egress}

## 1. Ingress — PII é tokenizada antes de o LLM ver (Doc 04, §7)

Enviamos um pedido com **CPF e conta**. Observe: o LLM (e os logs) veem apenas **tokens**; o valor real só reaparece no **egress**, reidratado para o titular autenticado.

In [ ]:
msg = 'Ola, sou a cliente Ana Souza, CPF 529.982.247-25, quero o saldo da conta corrente 45678-9.'
out = chat(msg, flow='authenticated', subject='ana')
print('ENTRADA   :', msg)
print('LLM viu   :', out['ingress']['text'])
print('entidades :', out['ingress']['entities_found'])
print('interno   :', out['answer_internal'])
print('entregue  :', out['answer'], '| rehydrated =', out['rehydrated'])

## 2. Ferramentas MCP isoladas — leitura ok, transferência bloqueada (Doc 04, §9-10)

A consulta de saldo (escopo `saldo:read`) é autorizada. Já a tentativa de **transferir** é negada — não há capacidade de escrita no catálogo (defesa estrutural contra o *confused deputy*).

In [ ]:
print('consultar_saldo (autorizado):')
print('  ', tool_call('consultar_saldo', 'saldo:read', {'conta_token': '[BR_ACCOUNT_TOKEN_demo0001]'}, 'ana'))
print('transferir (deve ser BLOQUEADO):')
print('  ', tool_call('transferir', 'transfer:write', {'para': '99999-9'}, 'atacante'))

## 3. Métricas de proteção: detecção de PII e latência (Doc 04, §6)

Rodamos a detecção sobre `data/pii_samples` (via `/anonymize` no modo LIVE, ou in-process no OFFLINE) e salvamos o relatório em `reports/lab_metrics.json`.

In [ ]:
import time

rows = [json.loads(l) for l in (ROOT / 'data' / 'pii_samples' / 'pii_samples.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]
expected = [r.get('entities', []) for r in rows]
predicted, lat = [], []
for r in rows:
    t0 = time.perf_counter()
    body = anonymize_call(r['text'], flow='authenticated')
    lat.append((time.perf_counter() - t0) * 1000)
    predicted.append(list(body.get('entities_found', {}).keys()))

cm = detection_confusion(expected, predicted)
report = {'mode': 'live' if LIVE else 'offline', 'samples': len(rows),
          'detection': cm.to_dict(), 'avg_latency_ms': round(sum(lat) / len(lat), 3)}
(ROOT / 'reports').mkdir(exist_ok=True)
(ROOT / 'reports' / 'lab_metrics.json').write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(report, indent=2, ensure_ascii=False))

In [ ]:
import matplotlib.pyplot as plt

d = cm.to_dict()
keys = ['precision', 'recall', 'f1', 'leak_rate']
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(keys, [d[k] for k in keys], color=['#3b7dd8', '#2e9e5b', '#8e44ad', '#c0392b'])
ax.set_ylim(0, 1)
ax.set_title('Deteccao de PII — metricas (Aula 03)')
for i, k in enumerate(keys):
    ax.text(i, d[k] + 0.02, f'{d[k]:.2f}', ha='center')
plt.tight_layout()
plt.show()

## 4. Minimização por fluxo (Doc 04, §12)

O mesmo texto recebe tratamentos distintos conforme a criticidade: o fluxo `faq` **remove** toda PII (não precisa dela); o `authenticated` **tokeniza** CPF/conta (reversível para o titular).

In [ ]:
msg = 'Meu CPF e 529.982.247-25 e minha conta corrente 45678-9.'
print('faq           ->', anonymize_call(msg, flow='faq')['text'])
print('authenticated ->', anonymize_call(msg, flow='authenticated')['text'])

## ✅ Resumo e gancho para a Aula 04

Integramos **anonimização (Presidio) + cofre de tokens + MCP isolado** no pipeline Zero Trust end-to-end (Figura 1): o dado bruto nunca chega ao modelo, a ferramenta é somente leitura e o abuso é bloqueado. Medimos as evidências (detecção, *leak rate*, latência). Na **Aula 04**, transformamos essas evidências em **testes de segurança e auditoria** automatizados (red teaming, trilhas imutáveis).